In [1]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

## Load dataset generated by CovaS

In [2]:
def calculate_macro_tpr_fpr(voting_cm):
    num_classes = voting_cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        TP = voting_cm[i, i]
        FN = np.sum(voting_cm[i, :]) - TP
        FP = np.sum(voting_cm[:, i]) - TP
        TN = np.sum(voting_cm) - (TP + FN + FP)

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

        tpr_list.append(TPR)
        fpr_list.append(FPR)

    macro_tpr = np.mean(tpr_list)
    macro_fpr = np.mean(fpr_list)

    return macro_tpr, macro_fpr

train = pd.read_csv('/dis/DS/hungnt/CICModbus2023/train_shap_58_600.csv')
test = pd.read_csv('/dis/DS/hungnt/CICModbus2023/test_shap_58_600.csv')

In [3]:
X_train = train.drop(['Label'], axis=1)
y_train = train['Label']
X_test = test.drop(['Label'], axis=1)
y_test = test['Label']

In [4]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

## XGBoost

In [5]:
xgb_params = {
    'tree_method': 'gpu_hist',
    'predictor': 'gpu_predictor',
    'max_depth': 15,
    'n_estimators': 3000,
    'learning_rate': 0.2,
    'eval_metric': ['mlogloss', 'merror'],
    'objective': 'multi:softprob',
    'num_class': len(y_train.unique()),
    'booster': 'gbtree',
    'random_state': 42,
    'early_stopping_rounds': 50
}

train_weight = compute_sample_weight("balanced", y_train_split)
val_weight   = compute_sample_weight("balanced", y_val_split)

print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)

xgb_model.fit(
    X_train_split, y_train_split,
    sample_weight=train_weight,
    eval_set=[(X_val_split, y_val_split)],
    sample_weight_eval_set=[val_weight],
    verbose=False
)
print("XGBClassifier Finished")

xgb_start_time = time.time()
xgb_prediction = xgb_model.predict(X_test)
xgb_end_time = time.time()
xgb_time = xgb_end_time - xgb_start_time
xgb_acc = sklearn.metrics.accuracy_score(y_test, xgb_prediction)
xgb_precision = sklearn.metrics.precision_score(y_test, xgb_prediction, average='macro')
xgb_f1 = sklearn.metrics.f1_score(y_test, xgb_prediction, average='macro')
xgb_recall = sklearn.metrics.recall_score(y_test, xgb_prediction, average='macro')
xgb_cm = sklearn.metrics.confusion_matrix(y_test, xgb_prediction)

print("XGBoost report:")
print("XGBoost Time:", xgb_time)
print("XGBoost Accuracy:", xgb_acc)
print("XGBoost Precision:", xgb_precision)
print("XGBoost F1:", xgb_f1)
print("XGBoost Recall:", xgb_recall)
print("XGBoost CM:\n", xgb_cm)
xgb_tpr, xgb_fpr = calculate_macro_tpr_fpr(xgb_cm)
print(f'XGBoost Macro-average TPR: {xgb_tpr}')
print(f'XGBoost Macro-average FPR: {xgb_fpr}')
print(classification_report(y_test, xgb_prediction, digits=4))

XGBClassifier Starting
XGBClassifier Finished
XGBoost report:
XGBoost Time: 0.08182549476623535
XGBoost Accuracy: 0.9437404967055246
XGBoost Precision: 0.9477259540031194
XGBoost F1: 0.9481993126828908
XGBoost Recall: 0.9489862773573132
XGBoost CM:
 [[591   4   0   0   0   0   2   2   1]
 [  1 506   3   0   0  11   1   0  29]
 [  0  11 539   0   0   7   0   0  43]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  2  17   0   0   0 390   8   0   4]
 [  0  12   0   0   0   5 430   0   0]
 [  1   0   0   0   2   0   0 450   0]
 [  1  13  28   0   0  13   0   0 337]]
XGBoost Macro-average TPR: 0.9489862773573132
XGBoost Macro-average FPR: 0.007111355240895515
              precision    recall  f1-score   support

           0     0.9916    0.9850    0.9883       600
           1     0.8988    0.9183    0.9084       551
           2     0.9456    0.8983    0.9214       600
           3     1.0000    1.0000    1.0000        26
           4     0.9956    0.9978

## ExtraTree

In [6]:
et_params = {
    "n_estimators": 468,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
et_prediction = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")

et_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=et_prediction)
et_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=et_prediction)
et_fp = et_cm[0, 1]
print("ExtraTrees report:")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees CM:\n", et_cm)
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')
print(classification_report(y_test, et_prediction, digits=4))

ExtraTreesClassifier Starting
ExtraTreesClassifier Finished
ExtraTrees report:
ExtraTrees Time: 0.3122375011444092
ExtraTrees Accuracy: 0.9526102382159148
ExtraTrees Precision: 0.9566703163546941
ExtraTrees F1: 0.9573197109021115
ExtraTrees Recall: 0.9593329954191133
ExtraTrees CM:
 [[592   3   0   0   0   2   2   1   0]
 [  6 528   3   0   0   7   0   0   7]
 [  0  15 513   0   0  20   0   0  52]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  6  11   0   0   0 402   1   0   1]
 [  4   5   0   0   0   6 431   0   1]
 [  2   0   0   0   2   0   0 449   0]
 [  0   8  13   0   0   7   1   0 363]]
ExtraTrees Macro-average TPR: 0.9593329954191133
ExtraTrees Macro-average FPR: 0.005988833998132232
              precision    recall  f1-score   support

           0     0.9705    0.9867    0.9785       600
           1     0.9263    0.9583    0.9420       551
           2     0.9698    0.8550    0.9088       600
           3     1.0000    1.0000    1.0000    

In [11]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
et_params = {
    "n_estimators": 70,        # sẽ thay trong vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

START, END, STEP = 460, 480, 1  # có thể đổi STEP=1 để quét mịn

print("ExtraTreesClassifier Sweep Starting")

records = []
best_key = None
best_model, best_cm, best_pred = None, None, None
best_n, best_time = None, None

for n in range(START, END + 1, STEP):
    params = et_params.copy()
    params["n_estimators"] = n

    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_test, pred)
    precision = sklearn.precision_score(y_test, pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_test, pred, average='macro')
    recall = sklearn.recall_score(y_test, pred, average='macro')
    cm = sklearn.confusion_matrix(y_test, pred)
    et_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": et_fp
    })

    key = (acc, f1, recall, -n)  # tie-break
    if best_key is None or key > best_key:
        best_key = key
        best_model, best_cm, best_pred = model, cm, pred
        best_n, best_time = n, t1 - t0

print("ExtraTreesClassifier Sweep Finished")

# === Tổng hợp kết quả ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], 
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === Report tốt nhất ===
et_acc = sklearn.accuracy_score(y_test, best_pred)
et_precision = sklearn.precision_score(y_test, best_pred, average='macro', zero_division=0)
et_f1 = sklearn.f1_score(y_test, best_pred, average='macro')
et_recall = sklearn.recall_score(y_test, best_pred, average='macro')
et_cm = best_cm
et_fp = et_cm[0, 1] if et_cm.shape == (2, 2) else None
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)

print("\n===== Best ExtraTrees report =====")
print(f"Best n_estimators: {best_n}")
print("ExtraTrees Time:", best_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

# df_results.to_csv("./et_sweep_n_estimators_30_500.csv", index=False)


ExtraTreesClassifier Sweep Starting
ExtraTreesClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0           468  0.952610  0.957320      0.959333  0.298490
1           470  0.952610  0.957320      0.959333  0.310467
2           471  0.952610  0.957320      0.959333  0.324132
3           472  0.952610  0.957320      0.959333  0.307253
4           474  0.952610  0.957320      0.959333  0.328799
5           475  0.952610  0.957320      0.959333  0.311044
6           476  0.952610  0.957320      0.959333  0.379356
7           477  0.952610  0.957320      0.959333  0.314956
8           478  0.952610  0.957320      0.959333  0.299109
9           473  0.952357  0.957095      0.959131  0.286770

===== Best ExtraTrees report =====
Best n_estimators: 468
ExtraTrees Time: 0.298490047454834
ExtraTrees Accuracy: 0.9526102382159148
ExtraTrees Precision: 0.9566703163546941
ExtraTrees F1: 0.9573197109021115
ExtraTrees Recall: 0

## RandomForest

In [7]:
rf_params = {
    "n_estimators": 908,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
rf_prediction = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")

rf_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=rf_prediction)
rf_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=rf_prediction)
rf_fp = rf_cm[0, 1]
print("RandomForest report:")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest CM: \n", rf_cm)
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest report:
RandomForest Time: 0.6698720455169678
RandomForest Accuracy: 0.9614799797263052
RandomForest Precision: 0.9647739569781558
RandomForest F1: 0.9650798084878411
RandomForest Recall: 0.9662486224301651
RandomForest CM: 
 [[596   2   1   0   1   0   0   0   0]
 [  0 533   1   0   0   4   0   0  13]
 [  0  14 534   0   0   9   0   0  43]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  2  15   0   0   0 402   1   0   1]
 [  2   5   0   0   0   4 435   0   1]
 [  2   0   0   0   2   0   0 449   0]
 [  0  11  12   0   0   4   1   0 364]]
RandomForest Macro-average TPR: 0.9662486224301651
RandomForest Macro-average FPR: 0.004868360856956591


In [12]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu bạn đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    """
    cm: confusion_matrix (C x C)
    Trả về (macro_TPR, macro_FPR)
    """
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
rf_params = {
    "n_estimators": 60,        # sẽ bị thay mỗi vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

START, END, STEP = 900, 920, 1   # có thể đổi STEP=1 nếu muốn quét mịn hơn

print("RandomForestClassifier Sweep Starting")

records = []
best_key = None  # (acc, f1, recall, -n_estimators) để có tie-break rõ ràng
best_model = None
best_cm = None
best_pred = None
best_n = None
best_time = None

for n in range(START, END + 1, STEP):
    params = rf_params.copy()
    params["n_estimators"] = n

    model = RandomForestClassifier(**params)
    model.fit(X=X_train, y=y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_true=y_test, y_pred=pred)
    precision = sklearn.precision_score(y_true=y_test, y_pred=pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_true=y_test, y_pred=pred, average='macro')
    recall = sklearn.recall_score(y_true=y_test, y_pred=pred, average='macro')
    cm = sklearn.confusion_matrix(y_true=y_test, y_pred=pred)
    # Lưu ý: rf_fp = cm[0, 1] chỉ có ý nghĩa khi bài toán nhị phân và label 0/1 nằm đúng trục
    rf_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": rf_fp
    })

    # Tie-break: ưu tiên acc -> f1 -> recall; nếu vẫn hòa, chọn n_estimators nhỏ hơn
    key = (acc, f1, recall, -n)
    if (best_key is None) or (key > best_key):
        best_key = key
        best_model = model
        best_cm = cm
        best_pred = pred
        best_n = n
        best_time = t1 - t0

print("RandomForestClassifier Sweep Finished")

# === Tổng hợp kết quả thành DataFrame và in Top-10 theo Accuracy ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === In report mô hình tốt nhất (giữ format bạn đang dùng) ===
rf_acc = sklearn.accuracy_score(y_true=y_test, y_pred=best_pred)
rf_precision = sklearn.precision_score(y_true=y_test, y_pred=best_pred, average='macro', zero_division=0)
rf_f1 = sklearn.f1_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_recall = sklearn.recall_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_cm = best_cm
rf_fp = rf_cm[0, 1] if rf_cm.shape == (2, 2) else None
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)

print("\n===== Best RandomForest report =====")
print(f"Best n_estimators: {best_n}")
print("RandomForest Time:", best_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Sweep Starting
RandomForestClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0           908  0.961480  0.965080      0.966249  0.553481
1           909  0.961480  0.965080      0.966249  0.473392
2           910  0.961227  0.964863      0.966063  0.429809
3           911  0.961227  0.964863      0.966063  0.567248
4           912  0.961227  0.964863      0.966063  0.457273
5           907  0.961227  0.964854      0.966063  0.448331
6           904  0.960973  0.964638      0.965878  0.408695
7           905  0.960973  0.964638      0.965878  0.557942
8           906  0.960973  0.964638      0.965878  0.404766
9           913  0.960973  0.964638      0.965878  0.436527

===== Best RandomForest report =====
Best n_estimators: 908
RandomForest Time: 0.5534811019897461
RandomForest Accuracy: 0.9614799797263052
RandomForest Precision: 0.9647739569781558
RandomForest F1: 0.9650798084878411
Rando

# lightgbm

In [5]:
import time
import lightgbm as lgb

lgbm_params = {
    'boosting_type': 'gbdt',
    'objective': 'multiclass',
    'num_class': int(y_train_split.nunique()),
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 3000,
    'device_type': 'cpu',
    'metric': ['multi_logloss', 'multi_error'],
    'random_state': 42,
    'class_weight': 'balanced'
}

print("LGBMClassifier Starting")
lgbm_model = lgb.LGBMClassifier(**lgbm_params)

lgbm_model.fit(
    X_train_split, y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    eval_metric=['multi_logloss', 'multi_error'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    verbose=False
)

lgbm_start_time = time.time()
lgbm_prediction = lgbm_model.predict(X_test)
lgbm_end_time = time.time()
lgbm_time = lgbm_end_time - lgbm_start_time
print("LGBMClassifier Finished")

lgbm_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=lgbm_prediction)
lgbm_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=lgbm_prediction)

print("LightGBM report:")
print("LightGBM Time:", lgbm_time)
print("LightGBM Accuracy:", lgbm_acc)
print("LightGBM Precision:", lgbm_precision)
print("LightGBM F1:", lgbm_f1)
print("LightGBM Recall:", lgbm_recall)
print("LightGBM CM:\n", lgbm_cm)
lgbm_tpr, lgbm_fpr = calculate_macro_tpr_fpr(lgbm_cm)
print(f'LightGBM Macro-average TPR: {lgbm_tpr}')
print(f'LightGBM Macro-average FPR: {lgbm_fpr}')
print(classification_report(y_test, lgbm_prediction, digits=4))

LGBMClassifier Starting


/home/soc/miniconda3/envs/soc/lib/python3.8/site-packages/lightgbm/sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "


LGBMClassifier Finished
LightGBM report:
LightGBM Time: 0.04966855049133301
LightGBM Accuracy: 0.9442473390775469
LightGBM Precision: 0.9482076053833196
LightGBM F1: 0.9486320348364257
LightGBM Recall: 0.9492969560157548
LightGBM CM:
 [[594   2   0   0   1   1   2   0   0]
 [  2 505   4   0   0  10   2   0  28]
 [  0  10 539   0   0   7   0   0  44]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  0  18   0   0   0 394   6   0   3]
 [  0  12   0   0   0   6 429   0   0]
 [  1   0   0   0   2   0   0 450   0]
 [  1  14  31   0   0  12   0   0 334]]
LightGBM Macro-average TPR: 0.9492969560157548
LightGBM Macro-average FPR: 0.007052184043789931
              precision    recall  f1-score   support

           0     0.9933    0.9900    0.9917       600
           1     0.9002    0.9165    0.9083       551
           2     0.9390    0.8983    0.9182       600
           3     1.0000    1.0000    1.0000        26
           4     0.9934    0.9978    0.9956   

# catboost

In [26]:
import time
from catboost import CatBoostClassifier

cat_params = {
    'task_type': 'GPU',
    'iterations': 3000,
    'depth': 5,
    'learning_rate': 0.2,
    'loss_function': 'MultiClass',
    'eval_metric': 'MultiClass',
    'random_seed': 42,
    'od_type': 'Iter', 
    'od_wait': 50,
    'verbose': False,
    'auto_class_weights': 'Balanced'
}

print("CatBoostClassifier Starting")
cat_model = CatBoostClassifier(**cat_params)
cat_model.fit(
    X_train_split, y_train_split,
    eval_set=(X_val_split, y_val_split),
    use_best_model=True,
    verbose=False
)

cat_start_time = time.time()
cat_prediction = cat_model.predict(X_test).astype(int).ravel()
cat_end_time = time.time()
cat_time = cat_end_time - cat_start_time
print("CatBoostClassifier Finished")

cat_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=cat_prediction)
cat_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=cat_prediction)

print("CatBoost report:")
print("CatBoost Time:", cat_time)
print("CatBoost Accuracy:", cat_acc)
print("CatBoost Precision:", cat_precision)
print("CatBoost F1:", cat_f1)
print("CatBoost Recall:", cat_recall)
print("CatBoost CM:\n", cat_cm)
cat_tpr, cat_fpr = calculate_macro_tpr_fpr(cat_cm)
print(f'CatBoost Macro-average TPR: {cat_tpr}')
print(f'CatBoost Macro-average FPR: {cat_fpr}')
print(classification_report(y_test, cat_prediction, digits=4))

CatBoostClassifier Starting
CatBoostClassifier Finished
CatBoost report:
CatBoost Time: 0.02089095115661621
CatBoost Accuracy: 0.9325899645210339
CatBoost Precision: 0.9371502934429573
CatBoost F1: 0.9377833596978383
CatBoost Recall: 0.939156300612484
CatBoost CM:
 [[594   3   0   0   1   0   2   0   0]
 [  3 482   1   0   0  16   3   0  46]
 [  0  11 531   0   0   8   0   0  50]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 452   0   1   0   3]
 [  1  12   0   0   0 394   9   0   5]
 [  1   8   0   0   0  10 428   0   0]
 [  2   1   0   0   2   0   0 448   0]
 [  1  23  29   0   0  14   0   0 325]]
CatBoost Macro-average TPR: 0.939156300612484
CatBoost Macro-average FPR: 0.008496350300632567
              precision    recall  f1-score   support

           0     0.9867    0.9900    0.9884       600
           1     0.8926    0.8748    0.8836       551
           2     0.9465    0.8850    0.9147       600
           3     1.0000    1.0000    1.0000        26
           4    

In [8]:
# ==============================================================================
# TỔNG KẾT KẾT QUẢ CÁC MÔ HÌNH
# ==============================================================================
print(f"\n{'='*60}\nTỔNG KẾT KẾT QUẢ CÁC MÔ HÌNH - TabDiff\n{'='*60}")

results = {
    'Model': ['XGBoost', 'RandomForest', 'ExtraTrees'],
    'Accuracy': [xgb_acc, rf_acc, et_acc],
    'Precision': [xgb_precision, rf_precision, et_precision],
    'F1-Score': [xgb_f1, rf_f1, et_f1],
    'Recall': [xgb_recall, rf_recall, et_recall],
    'TPR': [xgb_tpr, rf_tpr, et_tpr],
    'FPR': [xgb_fpr, rf_fpr, et_fpr],
    'Time (s)': [xgb_time, rf_time, et_time]
}

results_df = pd.DataFrame(results)

# Format các cột số
for col in ['Accuracy', 'Precision', 'F1-Score', 'Recall', 'TPR', 'FPR']:
    results_df[col] = results_df[col].apply(lambda x: f"{x:.4f}")
results_df['Time (s)'] = results_df['Time (s)'].apply(lambda x: f"{x:.4f}")

display(results_df)

print("\n" + "="*60)
print("HOÀN THÀNH!")
print("="*60)


TỔNG KẾT KẾT QUẢ CÁC MÔ HÌNH - TabDiff


,Model,Accuracy,Precision,F1-Score,Recall,TPR,FPR,Time (s)
0,XGBoost,0.9437,0.9477,0.9482,0.9490,0.9490,0.0071,0.0818
1,RandomForest,0.9615,0.9648,0.9651,0.9662,0.9662,0.0049,0.6699
2,ExtraTrees,0.9526,0.9567,0.9573,0.9593,0.9593,0.0060,0.3122



HOÀN THÀNH!
